# Cached decode
Does a KV cache change the logits?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.model import Decoder, DecoderConfig
model = Decoder(DecoderConfig(width=32, layers=2, context=64)).eval()
tokens = torch.randint(0, 258, (1, 32))
with torch.inference_mode():
    full = model(tokens)
    outputs, caches = model.decode(tokens[:, :16])
    tail = []
    for position in range(16, 32):
        logits, caches = model.decode(tokens[:, position:position + 1], caches)
        tail.append(logits)
    cached = torch.cat([outputs, *tail], dim=1)
print('maximum difference:', (full - cached).abs().max().item())
print('cache bytes:', sum(cache.bytes for cache in caches))
torch.testing.assert_close(full, cached)

In [ ]:
from pythonic.generation import prefill
from pythonic.experiments import benchmark
for chunk in (4, 16, 64):
    with torch.inference_mode():
        timing = benchmark(lambda: prefill(model, tokens, chunk), torch.device('cpu'), repeats=3 if smoke else 10)
    print(chunk, 'chunk size:', round(timing.median_ms, 3), 'ms')

Each request owns its caches. Decode appends new keys and values while keeping the original tensors unchanged.